<a href="https://colab.research.google.com/github/abhash-subedi25/Forex_bot/blob/main/computer_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clone the official repository
!git clone https://github.com/facebookresearch/segment-anything-2.git
%cd segment-anything-2

# Install SAM 2 in editable mode along with required utilities
!pip install -e .
!pip install supervision opencv-python matplotlib

Cloning into 'segment-anything-2'...
remote: Enumerating objects: 1107, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 1107 (delta 13), reused 7 (delta 7), pack-reused 1089 (from 2)
Receiving objects: 100% (1107/1107), 134.85 MiB | 22.94 MiB/s, done.
Resolving deltas: 100% (385/385), done.
/content/segment-anything-2
Obtaining file:///content/segment-anything-2
  Installing build dependencies ... canceledERROR: Operation cancelled by user
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.7/409.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 23.5 MB/s eta 0:00:00


In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)

# Download the SAM 2.1 Hiera Large model weights
!wget -P checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

--2026-10-10 05:30:14--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.249.182.33, 13.249.182.81, 13.249.182.39, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.249.182.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 898083611 (856M) [application/vnd.snesdev-page-table]
Saving to: ‘checkpoints/sam2.1_hiera_large.pt’

sam2.1_hiera_large. 100%[===================>] 856.48M  87.3MB/s    in 5.3s    

2026-10-10 05:30:20 (161 MB/s) - ‘checkpoints/sam2.1_hiera_large.pt’ saved [898083611/898083611]



In [ ]:
import torch
import numpy as np
import cv2
from sam2.build_sam import build_sam2_video_predictor

# 1. Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load model configuratio-n and weights
checkpoint = "checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

predictor = build_sam2_video_predictor(model_cfg, checkpoint, device=device)

# 3. Upload your video via Colab's file picker or set your video path
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0] # Gets the name of the uploaded mp4 file

# 4. Initialize inference state directly from the MP4 file
inference_state = predictor.init_state(video_path)
print("Inference state initialized successfully from MP4!")

Using device: cuda


Saving 2026-10-10 11-00-58.mp4 to 2026-10-10 11-00-58 (1).mp4
Inference state initialized successfully from MP4!


In [ ]:
!pip install decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 107.9 MB/s eta 0:00:00


In [ ]:
import numpy as np

# Define prompt parameters for the first frame (Frame 0)
ann_frame_idx = 0
ann_obj_id = 1    # Unique ID for our object
# Change these (x, y) coordinates to point to the object you want to track in your video
points = np.array([[300, 300]], dtype=np.float32)
labels = np.array([1], np.int32)                    # 1 means a positive click (object here)

# Add the point prompt to initialize tracking
_, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=ann_obj_id,
    points=points,
    labels=labels,
)

print("Prompt added! Object ID:", out_obj_ids)

Prompt added! Object ID: [1]


In [ ]:
import cv2
import os

# Run propagation across the video
video_segments = {}
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, out_obj_id in enumerate(out_obj_ids)
    }

print("Propagation complete! Rendering video...")

# Read original video info to recreate output video
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_video_path = "output_tracked.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # If masks exist for this frame, overlay them
    if frame_idx in video_segments:
        for obj_id, mask in video_segments[frame_idx].items():
            # mask is shape (1, H, W), squeeze it
            mask = mask.squeeze() > 0.0
            if mask.any():
                # Color the mask overlay (e.g., bright green tint)
                color_mask = np.zeros_info = frame
                frame[mask] = frame[mask] * 0.5 + np.array([0, 255, 0], dtype=np.uint8) * 0.5

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print(f"Saved tracked video as {output_video_path}!")

propagate in video: 100%|██████████| 109/109 [01:50<00:00,  1.01s/it]


Propagation complete! Rendering video...
Saved tracked video as output_tracked.mp4!
